In [1]:
!pip -q uninstall -y numpy pandas pyarrow datasets transformers accelerate evaluate sentencepiece protobuf scipy scikit-learn rouge_score sacrebleu
!pip -q install -U pip setuptools wheel

!pip -q install --no-cache-dir --force-reinstall \
  "numpy==2.1.3" "pandas==2.2.2" "pyarrow==16.1.0"

!pip -q install --no-cache-dir --force-reinstall "protobuf<6"

!pip -q install --no-cache-dir --force-reinstall \
  "transformers==4.44.2" "accelerate>=0.35.0" "datasets==2.21.0" "evaluate==0.4.3" \
  "sentencepiece" "scikit-learn" "scipy" "rouge_score" "sacrebleu"

print("Installed. Restart runtime before continuing if needed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cufflinks 0.17.3 requires numpy>=1.9.2, which is not installed.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
pandas-gbq 0.30.0 requires numpy>=1.18.1, which is not installed.
pandas-gbq 0.30.0 requires pandas>=1.1.4, which is not installed.
pandas-gbq 0.30.0 requires pyarrow>=4.0.0, which is not installed.
bigframes 2.38.0 requires numpy>=1.24.0, which is not installed.
bigframes 2.38.0 requires pandas>=1.5.3, which is not installed.
bigframes 2.38.0 requires pyarrow>=15.0.2, which is not installed.
spacy 3.8.11 requires numpy>=1.19.0; python_version >= "3.9", which is not installed.
osqp 1.1.1 requires numpy>=1.7, which is not installed.
osq

In [4]:

!pip uninstall -y torch torchvision torchaudio
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130 --force-reinstall

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/nightly/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 304.5 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 302.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 306.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 243.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 285.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 407.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1

In [1]:


import os
import re
import random
import json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
import evaluate

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA L4


In [4]:
DATA_PATH = "/content/drive/MyDrive/IRP_ChatBot/dataset/empathetic_dialogues/emotion-emotion_69k.csv"
OUT_DIR   = "/content/drive/MyDrive/IRP_ChatBot/models/empathetic_generator_t5_v2"

os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = "google/flan-t5-base"

MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 64

SEED = 42
EPOCHS = 3
LR = 3e-5

BATCH_TR = 8
BATCH_EV = 8
GRAD_ACCUM = 2

WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
PATIENCE = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Model:", MODEL_NAME)
print("Output directory:", OUT_DIR)

Model: google/flan-t5-base
Output directory: /content/drive/MyDrive/IRP_ChatBot/models/empathetic_generator_t5_v2


In [9]:
df = pd.read_csv(DATA_PATH)

context_col = "empathetic_dialogues"
response_col = "labels"
emotion_col = "emotion"

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

Original shape: (64636, 7)
Columns: ['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels', 'Unnamed: 5', 'Unnamed: 6']


In [10]:
df = df.dropna(subset=[context_col, response_col, emotion_col]).copy()

df[context_col] = df[context_col].astype(str).str.strip()
df[response_col] = df[response_col].astype(str).str.strip()
df[emotion_col] = df[emotion_col].astype(str).str.strip()

# remove trailing Agent:
df[context_col] = df[context_col].str.replace(r"\bAgent\s*:\s*$", "", regex=True).str.strip()

# remove literal nan and weak rows
df = df[df[response_col].str.lower() != "nan"]
df = df[(df[context_col].str.len() >= 5) & (df[response_col].str.len() >= 15)]

print("Cleaned shape:", df.shape)
df.head(3)

Cleaned shape: (62550, 7)


,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.,Oh was this something that happened because of...,NaN,NaN


In [11]:
weak_patterns = [
    r"^ok[.!]?$",
    r"^okay[.!]?$",
    r"^i see[.!]?$",
    r"^oh[.!]?$",
    r"^oh really[.!]?$",
    r"^that'?s nice[.!]?$",
    r"^wow[.!]?$"
]

def is_weak_reply(text):
    t = text.strip().lower()
    for pat in weak_patterns:
        if re.match(pat, t):
            return True
    return False

before = len(df)
df = df[~df[response_col].apply(is_weak_reply)].copy()
after = len(df)

print("Removed weak replies:", before - after)
print("Remaining rows:", after)

Removed weak replies: 0
Remaining rows: 62550


In [12]:
pairs = []

for _, row in df.iterrows():
    emo = row[emotion_col]
    ctx = row[context_col]
    rsp = row[response_col]

    src = f"""You are a supportive mental health assistant.

The user's emotion is {emo}.
User message: {ctx}

Write exactly 2 sentences.
Sentence 1 should acknowledge the feeling naturally.
Sentence 2 should ask one gentle follow-up question.
Do not repeat phrases.
Do not give medical advice."""

    pairs.append((src, rsp))

print("Built pairs:", len(pairs))
print("\nExample input:\n")
print(pairs[0][0])
print("\nExample target:\n")
print(pairs[0][1])

Built pairs: 62550

Example input:

You are a supportive mental health assistant.

The user's emotion is sentimental.
User message: Customer :I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.

Write exactly 2 sentences.
Sentence 1 should acknowledge the feeling naturally.
Sentence 2 should ask one gentle follow-up question.
Do not repeat phrases.
Do not give medical advice.

Example target:

Was this a friend you were in love with, or just a best friend?


In [13]:
train_pairs, val_pairs = train_test_split(
    pairs,
    test_size=0.1,
    random_state=SEED
)

train_df = pd.DataFrame(train_pairs, columns=["source", "target"])
val_df   = pd.DataFrame(val_pairs, columns=["source", "target"])

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

print("Train size:", len(train_ds))
print("Val size  :", len(val_ds))

Train size: 56295
Val size  : 6255


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [14]:
def preprocess(batch):
    model_inputs = tokenizer(
        batch["source"],
        max_length=MAX_SOURCE_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

Map:   0%|          | 0/56295 [00:00<?, ? examples/s]

Map:   0%|          | 0/6255 [00:00<?, ? examples/s]

In [15]:
def count_empty_labels(ds, n=10000):
    empty = 0
    for i in range(min(n, len(ds))):
        if len(ds[i]["labels"]) == 0:
            empty += 1
    return empty

print("Empty labels in train:", count_empty_labels(train_tok))
print("Empty labels in val  :", count_empty_labels(val_tok))

Empty labels in train: 0
Empty labels in val  : 0


In [16]:
def count_empty_labels(ds, n=10000):
    empty = 0
    for i in range(min(n, len(ds))):
        if len(ds[i]["labels"]) == 0:
            empty += 1
    return empty

print("Empty labels in train:", count_empty_labels(train_tok))
print("Empty labels in val  :", count_empty_labels(val_tok))

Empty labels in train: 0
Empty labels in val  : 0


In [17]:
def count_empty_labels(ds, n=10000):
    empty = 0
    for i in range(min(n, len(ds))):
        if len(ds[i]["labels"]) == 0:
            empty += 1
    return empty

print("Empty labels in train:", count_empty_labels(train_tok))
print("Empty labels in val  :", count_empty_labels(val_tok))

Empty labels in train: 0
Empty labels in val  : 0


In [18]:
args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,

    evaluation_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,

    logging_strategy="steps",
    logging_steps=100,

    learning_rate=LR,
    per_device_train_batch_size=BATCH_TR,
    per_device_eval_batch_size=BATCH_EV,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,

    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,

    fp16=False,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=5,

    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [19]:
import evaluate
import numpy as np

# Define model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Define data_collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Define compute_metrics
metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds_np = np.clip(preds, 0, tokenizer.vocab_size - 1).astype(int)
    decoded_preds = tokenizer.batch_decode(preds_np, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a list of str for predictions and references
    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1000,2.967300,2.776183,14.945000,2.772000,13.457700,13.442900
2000,2.914200,2.736410,16.886900,3.451200,15.243100,15.235600
3000,2.898000,2.714415,17.054800,3.561100,15.547800,15.555100
4000,2.817400,2.703419,17.482200,3.794300,15.905200,15.912200
5000,2.803200,2.693092,17.523600,3.917700,15.889500,15.884500


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=5000, training_loss=2.914410516357422, metrics={'train_runtime': 6758.8957, 'train_samples_per_second': 24.987, 'train_steps_per_second': 1.561, 'total_flos': 1.0681738577842176e+16, 'train_loss': 2.914410516357422, 'epoch': 1.4210601108426886})

In [ ]:
best_dir = os.path.join(OUT_DIR, "best_model")
trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)

print("Saved best model to:", best_dir)

Saved best model to: /content/drive/MyDrive/IRP_ChatBot/models/empathetic_generator_t5_v2/best_model


In [20]:
def generate_reply(context, emotion=None):
    if emotion:
        src = f"""You are a supportive mental health assistant.

The user's emotion is {emotion}.
User message: {context}

Write exactly 2 sentences.
Sentence 1 should acknowledge the feeling naturally.
Sentence 2 should ask one gentle follow-up question.
Do not repeat phrases.
Do not give medical advice."""
    else:
        src = f"""You are a supportive mental health assistant.

User message: {context}

Write exactly 2 sentences.
Sentence 1 should acknowledge the feeling naturally.
Sentence 2 should ask one gentle follow-up question.
Do not repeat phrases.
Do not give medical advice."""

    inputs = tokenizer(
        [src],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LEN
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=6,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
            early_stopping=True
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [26]:
test_cases = [
    # Anger
    ("I am furious about how they treated me.", "anger"),
    ("Everything is making me so angry today.", "anger"),
    ("I can't stand this anymore, I'm really mad.", "anger"),
    ("Why does this keep happening to me? I'm so frustrated.", "anger"),

    # Disgust
    ("That was absolutely disgusting and I hated it.", "disgust"),
    ("I feel sick thinking about what I saw.", "disgust"),
    ("This situation makes me feel really uncomfortable.", "disgust"),

    # Fear
    ("I'm really scared about what might happen next.", "fear"),
    ("I feel anxious and worried about everything.", "fear"),
    ("I can't stop thinking something bad will happen.", "fear"),
    ("This situation is making me very nervous.", "fear"),

    # Joy
    ("I just got promoted at work and I’m so happy!", "joy"),
    ("Today has been such a wonderful day.", "joy"),
    ("I feel amazing and everything is going great.", "joy"),
    ("I’m really excited about what’s coming next!", "joy"),

    # Sadness
    ("I feel so lonely and empty inside.", "sadness"),
    ("Nothing seems to be going right for me.", "sadness"),
    ("I miss the way things used to be.", "sadness"),
    ("I feel like giving up on everything.", "sadness"),

    # Surprise
    ("I can't believe this happened, it's shocking!", "surprise"),
    ("Wow, I didn’t expect that at all!", "surprise"),
    ("That news completely surprised me.", "surprise"),

    # Neutral
    ("I finished my work for today.", "neutral"),
    ("I am just going about my normal routine.", "neutral"),
    ("Today was neither good nor bad.", "neutral"),
]

In [27]:
import pandas as pd

results = []

for text, emotion in test_cases:
    reply = generate_reply(text, emotion=emotion)

    results.append({
        "Input Text": text,
        "Emotion": emotion,
        "Generated Reply": reply
    })

df = pd.DataFrame(results)
df

,Input Text,Emotion,Generated Reply
0,I am furious about how they treated me.,anger,I'm sorry to hear that. How did they treat you?
1,Everything is making me so angry today.,anger,I'm sorry to hear that. What are you going to do?
2,"I can't stand this anymore, I'm really mad.",anger,I'm sorry to hear that. What happened?
3,Why does this keep happening to me? I'm so fru...,anger,I don't know. It's just that I haven't been ab...
4,That was absolutely disgusting and I hated it.,disgust,Oh no! I'm sorry to hear that.
5,I feel sick thinking about what I saw.,disgust,I'm sorry to hear that. What did you see?
6,This situation makes me feel really uncomforta...,disgust,I'm sorry to hear that. What happened?
7,I'm really scared about what might happen next.,fear,I'm sorry to hear that. What happened?
8,I feel anxious and worried about everything.,fear,I'm sorry to hear that. What are you worried a...
9,I can't stop thinking something bad will happen.,fear,I'm sorry to hear that. What happened?


In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

best_dir = "/content/drive/MyDrive/IRP_ChatBot/models/empathetic_generator_t5_v2/best_model"

tokenizer = AutoTokenizer.from_pretrained(best_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(best_dir).to(DEVICE)
model.eval()

print("Reloaded improved generator successfully.")

Reloaded improved generator successfully.
